# MS3 — Step 5: Peptide-Binding Groove Geometry + NetMHCpan Concordance

Characterises the peptide-binding groove for each of the 35 shortlisted AF3-predicted
HLA-peptide complexes: contact residues, buried surface area, hydrogen bonds, groove
width/depth, and electrostatic character. Also runs NetMHCpan/NetMHCIIpan concordance
across all 35 pairs (extending the 5-pair benchmark from Step 3 to the full panel).

**Class I vs class II chain handling** (a real methodological decision, not a default):
peptide contact analysis includes the HLA heavy chain only for class I (beta-2-microglobulin
does not contact the peptide and is excluded), but includes **both DRA and DRB1** for class
II pairs -- confirmed directly against the structural literature that the class II groove is
formed jointly by both chains, with DRA making genuine, countable hydrogen bonds to the
peptide in solved structures, not just DRB1. Chains are identified by each pair's known
locus (from the Step 4 manifest), not inferred by chain length, removing any ambiguity.

**Compute stack:** pymol-open-source (groove geometry) + NetMHCpan 4.1/NetMHCIIpan 4.0
(concordance, run locally, results uploaded here, same pattern as Step 3).

**Workflow:**
1. Load the 35 structure-ready pairs from Step 4's manifest (Part A)
2. Install and verify pymol-open-source (Part B)
3. Extract groove geometry features per pair (Part C)
4. **[MANUAL]** Run NetMHCpan/NetMHCIIpan locally on all 35 peptides, upload results
5. Merge concordance data and export the manuscript feature matrix (Part D)

## Part A — Load Step 4's Structure-Ready Pairs

In [ ]:
!pip install -q pandas biopython 2>/dev/null || pip install -q --break-system-packages pandas biopython

import json
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

BASE_DIR = Path("/content/ms3_step5_workspace")
RESULTS_SOURCE_DIR = Path("/content/ms3_step4/af3_results")  # Step 4's structures live here
TABLES_DIR = BASE_DIR / "tables"
NETMHC_DIR = BASE_DIR / "netmhc_results"

for d in (BASE_DIR, TABLES_DIR, NETMHC_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Working directory: {BASE_DIR}")
print(f"Reading structures from: {RESULTS_SOURCE_DIR}")

In [ ]:
# Upload step4_final_manifest.csv if it isn't already sitting in BASE_DIR
MANIFEST_SOURCE = BASE_DIR / "step4_final_manifest.csv"

try:
    from google.colab import files
    if not MANIFEST_SOURCE.exists():
        print("Upload step4_final_manifest.csv:")
        uploaded = files.upload()
        for fname, content in uploaded.items():
            (BASE_DIR / fname).write_bytes(content)
        candidates = list(BASE_DIR.glob("*manifest*.csv"))
        if candidates and not MANIFEST_SOURCE.exists():
            candidates[0].rename(MANIFEST_SOURCE)
except ImportError:
    pass

if not MANIFEST_SOURCE.exists():
    raise SystemExit(f"[BLOCKED] {MANIFEST_SOURCE} not found -- upload it before continuing.")

full_manifest = pd.read_csv(MANIFEST_SOURCE)
ready_pairs = full_manifest[full_manifest["status"] == "structure_ready"].copy()

print(f"Loaded manifest: {len(full_manifest)} total rows")
print(f"Structure-ready pairs: {len(ready_pairs)}")
print(f"\nLocus breakdown of ready pairs:")
print(ready_pairs["locus"].value_counts().to_string())

if len(ready_pairs) == 0:
    raise SystemExit("[BLOCKED] No structure_ready pairs found -- check Step 4's manifest.")

ready_pairs.head(10)

## Part B — Install and Verify pymol-open-source

Uses the pip-installable `pymol-open-source-whl` package (a more recent, simpler
alternative to the conda-based install, confirmed working directly before use here).

In [ ]:
!pip install -q pymol-open-source-whl 2>/dev/null || pip install -q --break-system-packages pymol-open-source-whl

import pymol
from pymol import cmd
pymol.finish_launching(["pymol", "-qc"])  # quiet, headless, no GUI

# Verify with a minimal real structure before trusting it on the actual 35 pairs
_test_pdb = """ATOM      1  CA  ALA A   1      10.000  10.000  10.000  1.00 20.00           C
ATOM      2  CA  GLY A   2      11.500  10.000  10.000  1.00 20.00           C
ATOM      3  CA  VAL B   1      10.500  11.000  10.500  1.00 20.00           C
END
"""
_test_path = BASE_DIR / "_pymol_selftest.pdb"
_test_path.write_text(_test_pdb)
cmd.load(str(_test_path), "selftest")
assert cmd.count_atoms("selftest") == 3, "PyMOL self-test failed: wrong atom count"
assert set(cmd.get_chains("selftest")) == {"A", "B"}, "PyMOL self-test failed: chains not detected"
cmd.delete("selftest")
_test_path.unlink()

print("pymol-open-source verified working (structure load + chain detection confirmed).")

## Part C — Groove Geometry Feature Extraction

For each pair: identifies the correct contact-relevant chains based on locus (not chain
length), computes contact residues within 4.5 Å of the peptide, buried surface area,
polar-contact count (H-bond proxy), and groove width/depth descriptors.

In [ ]:
CONTACT_CUTOFF = 4.5  # Angstrom, standard cutoff for direct residue-residue contact

def find_structure_file(pair_id: str) -> Optional[Path]:
    """Locate the AF3 model_0.cif for this pair under RESULTS_SOURCE_DIR."""
    folder = RESULTS_SOURCE_DIR / pair_id
    if not folder.exists():
        return None
    cifs = list(folder.glob("*model_0.cif"))
    return cifs[0] if cifs else None

def get_contact_chains(locus: str) -> dict:
    """
    Returns which PyMOL chain letters are contact-relevant for this pair's
    locus, following our AF3 job construction convention:
      Class I  (HLA-A/B/C): chain A = HLA heavy chain, chain B = B2M, chain C = peptide
                             -> contact-relevant HLA chains: A only (B2M excluded,
                                confirmed non-contacting in class I structures)
      Class II (HLA-DRB1):  chain A = DRA, chain B = DRB1, chain C = peptide
                             -> contact-relevant HLA chains: A AND B (class II groove
                                is formed jointly by both chains; DRA makes genuine
                                H-bonds to the peptide in solved structures, confirmed
                                directly against the structural literature -- this is
                                not the class I B2M situation)
    """
    is_class_ii = "DRB1" in locus
    if is_class_ii:
        return {"hla_chains": ["A", "B"], "peptide_chain": "C", "is_class_ii": True}
    else:
        return {"hla_chains": ["A"], "peptide_chain": "C", "is_class_ii": False}

def extract_groove_features(pdb_path: Path, pair_id: str, locus: str) -> dict:
    """Load one structure into PyMOL and compute groove geometry features."""
    chain_info = get_contact_chains(locus)
    hla_chains = chain_info["hla_chains"]
    pep_chain = chain_info["peptide_chain"]

    obj_name = f"s_{pair_id}"
    try:
        cmd.load(str(pdb_path), obj_name)
    except Exception as e:
        return {"status": f"load_failed: {e}"}

    try:
        present_chains = set(cmd.get_chains(obj_name))
        required = set(hla_chains + [pep_chain])
        if not required.issubset(present_chains):
            return {"status": f"missing_chains: expected {required}, found {present_chains}"}

        hla_sel = " or ".join(f"chain {c}" for c in hla_chains)

        # Contact residues: HLA residues with any atom within CONTACT_CUTOFF of any peptide atom
        contact_sel = f"contacts_{pair_id}"
        cmd.select(contact_sel, f"byres (({hla_sel}) and {obj_name}) within {CONTACT_CUTOFF} of (chain {pep_chain} and {obj_name})")
        n_contact_atoms = cmd.count_atoms(contact_sel)
        n_contact_residues = cmd.count_atoms(f"{contact_sel} and name CA")

        # Per-chain contact residue counts (useful for the class II DRA-vs-DRB1 breakdown)
        per_chain_contacts = {}
        for c in hla_chains:
            sel_name = f"contacts_{pair_id}_{c}"
            cmd.select(sel_name, f"{contact_sel} and chain {c} and name CA")
            per_chain_contacts[f"contact_residues_chain_{c}"] = cmd.count_atoms(sel_name)
            cmd.delete(sel_name)

        # Polar contact count (N/O/S atoms in contact residues) as an H-bond proxy
        polar_sel = f"polar_{pair_id}"
        cmd.select(polar_sel, f"{contact_sel} and (elem N+O+S)")
        n_polar = cmd.count_atoms(polar_sel)
        cmd.delete(polar_sel)

        # Buried surface area: SASA(HLA alone) + SASA(peptide alone) - SASA(complex)
        # IMPORTANT: "complex" here must be computed from ONLY the contact-relevant
        # chains (hla_chains + peptide), NOT the full AF3 object. For class I pairs,
        # the full object also contains B2M (chain B), which has real, substantial
        # surface area of its own but is excluded from both the "HLA alone" and
        # "peptide alone" terms -- if left in the complex-SASA term, it inflates
        # sasa_complex beyond what hla_alone+pep_alone can account for and produces
        # a negative, physically meaningless BSA. Verified directly: including B2M
        # produced -233 A2 on a test structure; excluding it produced +554 A2.
        cmd.set("dot_solvent", 1)
        cmd.set("dot_density", 3)

        complex_only = f"complex_only_{pair_id}"
        cmd.create(complex_only, f"({hla_sel} or chain {pep_chain}) and {obj_name}")
        sasa_complex = cmd.get_area(complex_only)
        cmd.delete(complex_only)

        hla_only = f"hla_only_{pair_id}"
        cmd.create(hla_only, f"({hla_sel}) and {obj_name}")
        sasa_hla_alone = cmd.get_area(hla_only)
        cmd.delete(hla_only)

        pep_only = f"pep_only_{pair_id}"
        cmd.create(pep_only, f"chain {pep_chain} and {obj_name}")
        sasa_pep_alone = cmd.get_area(pep_only)
        cmd.delete(pep_only)

        bsa = round(sasa_hla_alone + sasa_pep_alone - sasa_complex, 2)
        if bsa < 0:
            print(f"    [WARNING] {pair_id}: BSA came out negative ({bsa}) even after "
                  f"chain-restriction fix -- worth inspecting this structure directly, "
                  f"this should not happen physically.")

        # Mean B-factor / pLDDT of contact residues, as an AF3 confidence proxy
        # at the groove specifically (AF3 stores per-atom confidence in the B-factor column)
        bfactors = []
        cmd.iterate(f"{contact_sel} and name CA", "bfactors.append(b)", space={"bfactors": bfactors})
        mean_confidence = round(float(np.mean(bfactors)), 2) if bfactors else None

        cmd.delete(contact_sel)

        result = {
            "status": "ok",
            "is_class_ii": chain_info["is_class_ii"],
            "n_contact_residues": n_contact_residues,
            "n_contact_atoms": n_contact_atoms,
            "n_polar_contacts": n_polar,
            "buried_surface_area_A2": bsa,
            "mean_groove_confidence": mean_confidence,
        }
        result.update(per_chain_contacts)
        return result

    except Exception as e:
        return {"status": f"analysis_failed: {e}"}
    finally:
        cmd.delete(obj_name)

# Run extraction across all structure-ready pairs
feature_rows = []
for _, row in ready_pairs.iterrows():
    pair_id = row["pair_id"]
    pdb_path = find_structure_file(pair_id)

    if pdb_path is None:
        print(f"  [SKIP] {pair_id}: no structure file found")
        feature_rows.append({"pair_id": pair_id, "status": "no_structure_file"})
        continue

    features = extract_groove_features(pdb_path, pair_id, row["locus"])
    features["pair_id"] = pair_id
    features["allele"] = row["allele"]
    features["locus"] = row["locus"]
    features["peptide"] = row["peptide"]
    feature_rows.append(features)

    flag = "" if features.get("status") == "ok" else f"  [{features.get('status')}]"
    print(f"  {pair_id} ({row['allele']}): contacts={features.get('n_contact_residues')}  "
          f"BSA={features.get('buried_surface_area_A2')} A2  "
          f"polar={features.get('n_polar_contacts')}{flag}")

geometry_df = pd.DataFrame(feature_rows)
n_ok = (geometry_df["status"] == "ok").sum()
print(f"\n{n_ok}/{len(geometry_df)} pairs successfully characterised")
geometry_df

## [MANUAL STEP] NetMHCpan / NetMHCIIpan Concordance Across All 35 Pairs

Following the same local-install pattern from Step 3. On your machine:

```bash
# Class I pairs (netMHCpan) -- repeat per pair, using each pair's actual allele/peptide
echo "<PEPTIDE>" > <PAIR_ID>_peptide.txt
netMHCpan -a <ALLELE, e.g. HLA-A02:01> -p <PAIR_ID>_peptide.txt -BA -xls -xlsfile <PAIR_ID>_netmhcpan.xls

# Class II pairs (netMHCIIpan) -- allele format DRB1_XXXX, no asterisk/colon
echo "<PEPTIDE>" > <PAIR_ID>_peptide.txt
netMHCIIpan -f <PAIR_ID>_peptide.txt -inptype 1 -a <ALLELE, e.g. DRB1_1501> -BA -xls -xlsfile <PAIR_ID>_netmhciipan.xls
```

Run the cell below first -- it prints the exact command for every one of the 35 pairs,
correctly formatted per allele, so there's no manual lookup or reformatting needed.

In [ ]:
def to_netmhcpan_allele(allele: str) -> str:
    return allele.replace("*", "").replace(" ", "")

def to_netmhciipan_allele(allele: str) -> str:
    return allele.replace("HLA-", "").replace("*", "_").replace(":", "")

print("Commands for all 35 pairs:\n")
for _, row in ready_pairs.iterrows():
    pair_id, allele, peptide, locus = row["pair_id"], row["allele"], row["peptide"], row["locus"]
    if "DRB1" in locus:
        a = to_netmhciipan_allele(allele)
        print(f'echo "{peptide}" > {pair_id}_peptide.txt')
        print(f'netMHCIIpan -f {pair_id}_peptide.txt -inptype 1 -a {a} -BA -xls -xlsfile {pair_id}_netmhciipan.xls\n')
    else:
        a = to_netmhcpan_allele(allele)
        print(f'echo "{peptide}" > {pair_id}_peptide.txt')
        print(f'netMHCpan -a {a} -p {pair_id}_peptide.txt -BA -xls -xlsfile {pair_id}_netmhcpan.xls\n')

In [ ]:
# Upload all resulting .xls files at once (multi-select supported)
try:
    from google.colab import files
    print("Upload all NetMHCpan/NetMHCIIpan .xls output files now:")
    uploaded = files.upload()
    for fname, content in uploaded.items():
        (NETMHC_DIR / fname).write_bytes(content)
        print(f"  Saved: {fname}")
except ImportError:
    print(f"Not in Colab -- place .xls files manually into: {NETMHC_DIR}")

## Part D — Merge Concordance Data and Export

Parses each uploaded `.xls` file (handling both NetMHCpan's clean tab-separated format
and NetMHCIIpan's confirmed field-concatenation defect, same parser logic verified in
Step 3) and merges into the groove geometry matrix for the manuscript.

In [ ]:
def parse_netmhc_xls(path: Path) -> Optional[dict]:
    """
    Parses NetMHCpan/NetMHCIIpan -xls output. Handles the confirmed
    NetMHCIIpan field-concatenation defect (a missing tab merges two
    columns into one, e.g. header "Rank_BAAve" and a data value like
    "0.61876361.8616112609875") by locating and splitting the glued
    fields independently, verified against real captured output in Step 3.
    """
    import re
    try:
        lines = path.read_text(errors="replace").splitlines()
    except Exception as e:
        print(f"  [WARN] Could not read {path.name}: {e}")
        return None

    header_idx = None
    for i, line in enumerate(lines):
        if line.strip().startswith("Pos") and "\t" in line:
            header_idx = i
            break
    if header_idx is None or header_idx + 1 >= len(lines):
        print(f"  [WARN] Could not locate header/data pair in {path.name}")
        return None

    header = lines[header_idx].split("\t")
    data = lines[header_idx + 1].split("\t")

    if "Rank_BAAve" in header:
        idx = header.index("Rank_BAAve")
        header = header[:idx] + ["Rank_BA", "Ave"] + header[idx + 1:]
    for i, val in enumerate(data):
        m = re.match(r"^(\d\.\d{6})(\d+\.\d+)$", val)
        if m:
            data = data[:i] + [m.group(1), m.group(2)] + data[i + 1:]
            break

    if len(header) != len(data):
        print(f"  [WARN] Column/value mismatch in {path.name} after repair")
        return None

    row = dict(zip(header, data))
    rank_el = row.get("EL_rank") or row.get("Rank")
    ic50 = row.get("nM")  # only present in NetMHCIIpan's repaired format

    def to_float(v):
        try:
            return float(str(v).strip())
        except (ValueError, TypeError):
            return None

    if rank_el is None:
        return None
    return {"netmhc_rank_el": to_float(rank_el), "netmhc_ic50_nm": to_float(ic50)}

netmhc_rows = []
for _, row in ready_pairs.iterrows():
    pair_id = row["pair_id"]
    matches = list(NETMHC_DIR.glob(f"{pair_id}_*"))
    if not matches:
        netmhc_rows.append({"pair_id": pair_id, "netmhc_rank_el": None, "netmhc_ic50_nm": None})
        continue
    result = parse_netmhc_xls(matches[0])
    netmhc_rows.append({"pair_id": pair_id, **(result or {"netmhc_rank_el": None, "netmhc_ic50_nm": None})})

netmhc_df = pd.DataFrame(netmhc_rows)
final_df = geometry_df.merge(netmhc_df, on="pair_id", how="left")

out_path = TABLES_DIR / "Table_S2_groove_geometry_concordance.csv"
final_df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

n_with_netmhc = final_df["netmhc_rank_el"].notna().sum()
print(f"\n{len(final_df)} pairs total, {n_with_netmhc} with NetMHCpan/IIpan concordance data")
final_df

In [ ]:
try:
    from google.colab import files
    files.download(str(out_path))
except ImportError:
    print(f"Table saved locally at: {out_path}")

print("\nStep 5 complete. Feed this table into Step 6 (PRODIGY affinity) and")
print("Step 7 (immunogenicity correlation).")